# Local RAG Blueprint For Your Personal Book Library

This notebook implements a standard, non-agentic Retrieval-Augmented Generation (RAG) system for PDF books stored on your computer.

## What this design does

- Uses LangChain to load PDF pages and split them into retrieval chunks.
- Preserves original book and page metadata so answers can cite the right source.
- Builds embeddings locally with a lightweight sentence-transformer model. This keeps ingestion cheap and fast.
- Stores chunk vectors and metadata in a local Weaviate instance.
- Uses a hosted LLM from Hugging Face only for answer generation, not for indexing.

## Why this architecture fits your use case

- You do not want to host a local LLM, but you still want a fully local retrieval pipeline.
- You need metadata-aware retrieval such as `book_title = Book A` and page-level grounding.
- You want to learn LangChain without hiding the important retrieval and storage steps behind too many abstractions.

## Where LangChain helps here

- `PyMuPDFLoader` gives a clean document abstraction for page-level PDF loading.
- `RecursiveCharacterTextSplitter` handles chunking more cleanly than manual slicing.
- You still keep direct control over embeddings, Weaviate schema, and prompt construction.

## Recommended packages

- `langchain-community`: document loaders such as `PyMuPDFLoader`.
- `langchain-text-splitters`: text chunking utilities.
- `sentence-transformers`: local embedding model for semantic retrieval.
- `weaviate-client`: local vector database client.
- `huggingface_hub`: hosted inference access for DeepSeek or another HF model.
- `pandas`: inspect indexed chunks and retrieval results.
- `tqdm`: progress bars during indexing.

## Important limitation

If some PDFs are scanned images instead of text PDFs, add an OCR step before indexing. A common upgrade path is `ocrmypdf` or `unstructured`, but this notebook starts with text-based PDFs only.

In [1]:
# Run this once in the notebook if any package import is missing.
# %pip install -U langchain-community langchain-text-splitters pymupdf sentence-transformers weaviate-client huggingface_hub pandas tqdm\

from __future__ import annotations

import os
import re
import sys
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import time
import random
import pandas as pd
import weaviate
from huggingface_hub import InferenceClient
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm
from weaviate.classes.config import Configure, DataType, Property
from weaviate.classes.query import Filter, MetadataQuery

# Add the project root to the Python path so the notebook can reuse local helpers if needed.
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name.lower() == "notebook" else NOTEBOOK_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

try:
    from src import utils_private
except Exception:
    utils_private = None

# Central configuration for the local RAG pipeline.
HF_API_KEY = os.getenv("HF_API_KEY") or getattr(utils_private, "hf_api_key", None)
PDF_ROOT = PROJECT_ROOT / "books"
WEAVIATE_HOST = "127.0.0.1"
WEAVIATE_PORT = 8080
COLLECTION_NAME = "BookChunks"
EMBEDDING_MODEL_NAME = "BAAI/bge-small-en-v1.5"
CHAT_MODEL_NAME = "deepseek-ai/DeepSeek-V3-0324"
CHUNK_SIZE = 1200
CHUNK_OVERLAP = 200

print(f"Project root: {PROJECT_ROOT}")
print(f"PDF root exists: {PDF_ROOT.exists()}")
print(f"Hugging Face key loaded: {HF_API_KEY is not None}")

c:\Users\jason\miniconda3\envs\ai\Lib\site-packages\authlib\_joserfc_helpers.py:8: AuthlibDeprecationWarning: authlib.jose module is deprecated, please use joserfc instead.
It will be compatible before version 2.0.0.
  from authlib.jose import ECKey


Project root: C:\Program Files\Studying\coding\RAG_project
PDF root exists: True
Hugging Face key loaded: True


In [2]:
@dataclass
class ChunkRecord:
    """Represents one retrievable chunk tied back to its original book and page."""

    book_id: str
    book_title: str
    book_category: str
    source_path: str
    page_number: int
    chunk_index: int
    text: str


def normalize_book_title(file_path: Path) -> str:
    # Convert a PDF filename into a cleaner title for retrieval and display.
    return re.sub(r"[_-]+", " ", file_path.stem).strip()


def infer_book_category(file_path: Path, pdf_root: Path) -> str:
    # Infer a coarse category from the parent folder if the library is organized by folders.
    relative_parent = file_path.parent.relative_to(pdf_root) if file_path.parent != pdf_root else Path(".")
    if str(relative_parent) == ".":
        return "unknown"
    return relative_parent.parts[0]


def get_text_splitter(
    chunk_size: int = CHUNK_SIZE,
    chunk_overlap: int = CHUNK_OVERLAP,
 ) -> RecursiveCharacterTextSplitter:
    # Create the LangChain text splitter used for page-level chunking.
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )


def load_pdf_pages_with_langchain(pdf_path: Path):
    # Use LangChain's PyMuPDFLoader to return one document per PDF page with metadata attached.
    loader = PyMuPDFLoader(str(pdf_path))
    return loader.load()


def build_chunk_records(pdf_path: Path, pdf_root: Path) -> list[ChunkRecord]:
    # Turn one PDF into chunk records using LangChain for loading and chunk splitting.
    records: list[ChunkRecord] = []
    book_title = normalize_book_title(pdf_path)
    book_id = re.sub(r"\W+", "_", book_title.lower()).strip("_")
    book_category = infer_book_category(pdf_path, pdf_root)
    page_documents = load_pdf_pages_with_langchain(pdf_path)
    splitter = get_text_splitter()

    for page_document in page_documents:
        page_text = re.sub(r"\s+", " ", page_document.page_content).strip()
        if not page_text:
            continue

        page_number = int(page_document.metadata.get("page", 0)) + 1
        split_documents = splitter.create_documents(
            texts=[page_text],
            metadatas=[page_document.metadata],
        )

        for chunk_index, split_document in enumerate(split_documents, start=1):
            chunk_text = re.sub(r"\s+", " ", split_document.page_content).strip()
            if not chunk_text:
                continue

            records.append(
                ChunkRecord(
                    book_id=book_id,
                    book_title=book_title,
                    book_category=book_category,
                    source_path=str(pdf_path),
                    page_number=page_number,
                    chunk_index=chunk_index,
                    text=chunk_text,
                )
            )
    return records


def collect_chunk_records(pdf_root: Path) -> list[ChunkRecord]:
    # Walk the library folder and collect chunk records from every PDF file.
    pdf_files = sorted(pdf_root.rglob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError(f"No PDF files found under: {pdf_root}")

    all_records: list[ChunkRecord] = []
    for pdf_path in tqdm(pdf_files, desc="Parsing PDFs with LangChain"):
        all_records.extend(build_chunk_records(pdf_path, pdf_root))
    return all_records


def records_to_dataframe(records: list[ChunkRecord]) -> pd.DataFrame:
    # Convert chunk records to a DataFrame so you can inspect what will be indexed.
    return pd.DataFrame([record.__dict__ for record in records])

In [ ]:
# Lazy global handles keep the notebook responsive by loading models only when needed.
import random
from xmlrpc import client

_embedding_model = None

def get_embedding_model(model_name: str = EMBEDDING_MODEL_NAME) -> SentenceTransformer:
    # Load the embedding model on demand so indexing and query cells stay simple to call.
    global _embedding_model
    if _embedding_model is None:
        _embedding_model = SentenceTransformer(model_name)
    return _embedding_model


def embed_texts(texts: list[str], model_name: str = EMBEDDING_MODEL_NAME, batch_size: int = 32):
    # Convert text chunks or queries into normalized vectors for semantic retrieval.
    model = get_embedding_model(model_name)
    return model.encode(texts, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)


def connect_weaviate_local(host: str = WEAVIATE_HOST, port: int = WEAVIATE_PORT):
    # Connect to the local Weaviate instance that stores chunk vectors and metadata.
    return weaviate.connect_to_local(host=host, port=port)


def delete_collection(client, collection_name):
    # WARNING: This will delete all data in the collection. Use with caution.
    try:
        client.collections.delete(collection_name)
        print(f"Collection '{collection_name}' deleted.")
    except Exception as e:
        print(f"Error: {e}")


def create_or_get_collection(client, collection_name: str = COLLECTION_NAME, reset_collection: bool = False):
    # Create a collection with self-provided vectors so embeddings come from our local model.
    if reset_collection and client.collections.exists(collection_name):
        client.collections.delete(collection_name)

    if not client.collections.exists(collection_name):
        client.collections.create(
            name=collection_name,
            properties=[
                Property(name="book_id", data_type=DataType.TEXT),
                Property(name="book_title", data_type=DataType.TEXT),
                Property(name="book_category", data_type=DataType.TEXT),
                Property(name="source_path", data_type=DataType.TEXT),
                Property(name="page_number", data_type=DataType.INT),
                Property(name="chunk_index", data_type=DataType.INT),
                Property(name="text", data_type=DataType.TEXT),
            ],
            vector_config=Configure.Vectors.self_provided(),
        )

    return client.collections.get(collection_name)


def index_books(
    client,
    records,
    pdf_root: Path = PDF_ROOT,
    collection_name: str = COLLECTION_NAME,
    reset_collection: bool = False,
    embedding_model_name: str = EMBEDDING_MODEL_NAME,
    batch_size: int = 64,
    preview_rows: int = 5,
 ) -> pd.DataFrame:
    # Parse PDFs, embed chunk text, and store both vectors and metadata in Weaviate.

    vectors = embed_texts([record.text for record in records], model_name=embedding_model_name)

    try:
        collection = create_or_get_collection(client, collection_name=collection_name, reset_collection=reset_collection)
        with collection.batch.fixed_size(batch_size=batch_size) as batch:
            for record, vector in tqdm(list(zip(records, vectors)), desc="Indexing chunks", total=len(records)):
                batch.add_object(
                    properties={
                        "book_id": record.book_id,
                        "book_title": record.book_title,
                        "book_category": record.book_category,
                        "source_path": record.source_path,
                        "page_number": record.page_number,
                        "chunk_index": record.chunk_index,
                        "text": record.text,
                    },
                    vector=vector.tolist(),
                )
    finally:
        client.close()

    preview_df = records_to_dataframe(records)
    print(f"Indexed {len(records)} chunks into collection '{collection_name}'.")
    return preview_df


def build_filters(book_title: Optional[str] = None, book_category: Optional[str] = None):
    # Build optional metadata filters so retrieval can be scoped to a specific book or category.
    active_filter = None
    if book_title:
        active_filter = Filter.by_property("book_title").equal(book_title)
    if book_category:
        category_filter = Filter.by_property("book_category").equal(book_category)
        active_filter = category_filter if active_filter is None else active_filter & category_filter
    return active_filter


def search_chunks(
    query_text: str,
    collection_name: str = COLLECTION_NAME,
    book_title: Optional[str] = None,
    book_category: Optional[str] = None,
    top_k: int = 8,
 ) -> list[dict]:
    # Retrieve the most relevant chunks from Weaviate for a natural-language question.
    query_vector = embed_texts([query_text])[0].tolist()
    client = connect_weaviate_local()

    try:
        collection = client.collections.get(collection_name)
        response = collection.query.near_vector(
            near_vector=query_vector,
            filters=build_filters(book_title=book_title, book_category=book_category),
            limit=top_k,
            return_metadata=MetadataQuery(distance=True),
        )
    finally:
        client.close()

    hits: list[dict] = []
    for obj in response.objects:
        properties = obj.properties
        hits.append(
            {
                "book_title": properties["book_title"],
                "book_category": properties["book_category"],
                "page_number": properties["page_number"],
                "chunk_index": properties["chunk_index"],
                "text": properties["text"],
                "source_path": properties["source_path"],
                "distance": obj.metadata.distance,
            }
        )
    return hits


def find_relevant_books(query_text: str, top_k: int = 12) -> pd.DataFrame:
    # Aggregate retrieval hits to answer questions like 'which books talk about neural networks?'.
    hits = search_chunks(query_text, top_k=top_k)
    scores = defaultdict(lambda: {"match_count": 0, "best_distance": 1.0, "book_category": "unknown"})

    for hit in hits:
        book_stats = scores[hit["book_title"]]
        book_stats["match_count"] += 1
        book_stats["best_distance"] = min(book_stats["best_distance"], hit["distance"])
        book_stats["book_category"] = hit["book_category"]

    rows = [
        {
            "book_title": book_title,
            "book_category": values["book_category"],
            "match_count": values["match_count"],
            "best_distance": values["best_distance"],
        }
        for book_title, values in scores.items()
    ]
    return pd.DataFrame(rows).sort_values(["match_count", "best_distance"], ascending=[False, True]).reset_index(drop=True)


def find_relevant_pages(query_text: str, book_title: str, top_k: int = 10) -> pd.DataFrame:
    # Aggregate chunk hits by page to answer questions like 'which pages in Book A cover system design?'.
    hits = search_chunks(query_text, book_title=book_title, top_k=top_k)
    scores = defaultdict(lambda: {"match_count": 0, "best_distance": 1.0, "sample_text": ""})

    for hit in hits:
        page_stats = scores[hit["page_number"]]
        page_stats["match_count"] += 1
        page_stats["best_distance"] = min(page_stats["best_distance"], hit["distance"])
        if not page_stats["sample_text"]:
            page_stats["sample_text"] = hit["text"][:300]

    rows = [
        {
            "book_title": book_title,
            "page_number": page_number,
            "match_count": values["match_count"],
            "best_distance": values["best_distance"],
            "sample_text": values["sample_text"],
        }
        for page_number, values in scores.items()
    ]
    return pd.DataFrame(rows).sort_values(["match_count", "best_distance"], ascending=[False, True]).reset_index(drop=True)


def build_context_from_hits(hits: list[dict]) -> str:
    # Format retrieved chunks into a compact context block with citation-friendly headers.
    context_blocks = []
    for rank, hit in enumerate(hits, start=1):
        context_blocks.append(
            f"[Context {rank}] Book: {hit['book_title']} | Page: {hit['page_number']} | Chunk: {hit['chunk_index']}\n{hit['text']}"
        )
    return "\n\n".join(context_blocks)


def answer_question(
    query_text: str,
    book_title: Optional[str] = None,
    book_category: Optional[str] = None,
    top_k: int = 6,
    model_name: str = CHAT_MODEL_NAME,
 ) -> dict:
    # Retrieve evidence from Weaviate, then ask the hosted LLM to answer with book and page citations.
    hits = search_chunks(
        query_text=query_text,
        book_title=book_title,
        book_category=book_category,
        top_k=top_k,
    )

    if not hits:
        return {
            "answer": "No matching chunks were found in the vector database.",
            "citations": [],
            "raw_hits": [],
        }

    if not HF_API_KEY:
        raise ValueError("Hugging Face API key not found. Set HF_API_KEY or load it from src/utils_private.py")

    context_text = build_context_from_hits(hits)
    client = InferenceClient(api_key=HF_API_KEY, provider="auto")

    system_prompt = (
        "You are a grounded assistant for a private book library. "
        "Answer only from the provided context. "
        "If the context is insufficient, say so explicitly. "
        "When you cite evidence, mention the book title and page number."
    )

    user_prompt = f"""
    Question: {query_text}

    Book filter: {book_title or 'None'}
    Category filter: {book_category or 'None'}

    Context:
    {context_text}

    Please provide:
    1. A concise answer grounded in the context.
    2. A short citation list with book title and page number.
    3. A note when the evidence is weak or ambiguous.
    """

    completion = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
    )

    citations = [
        {
            "book_title": hit["book_title"],
            "page_number": hit["page_number"],
            "chunk_index": hit["chunk_index"],
            "distance": hit["distance"],
        }
        for hit in hits
    ]

    return {
        "answer": completion.choices[0].message["content"],
        "citations": citations,
        "raw_hits": hits,
    }

## How to use this notebook

1. Put your PDF files under a local folder such as `books/technical`, `books/non_fiction`, or `books/fiction`.
2. Make sure your Weaviate Docker container is running locally on port `8080`.
3. Run the import/config cell, then run the function-definition cells.
4. Run the usage cell below to index your library and test retrieval.

## Production-minded notes

- The indexing job and the query API should eventually be separate processes.
- Keep the vector schema stable because reindexing large libraries is expensive.
- Add OCR only when you confirm some PDFs are image-based.
- Add reranking later only if retrieval quality is not strong enough.

In [15]:
import pprint

In [16]:
# Example usage: run these lines step by step after you place PDFs under PROJECT_ROOT / 'books'.

# Preview parsed chunk records before indexing.
# sample_records = collect_chunk_records(PDF_ROOT)
# chunk_record = records_to_dataframe(sample_records)

# Build or rebuild the vector index.
# index_preview = index_books(pdf_root=PDF_ROOT, reset_collection=False)
# index_preview

# Ask which books discuss a topic across the whole library.
# relevant_books = find_relevant_books("Which books talk about neural networks?")
# relevant_books

# Ask which pages in one book mention a concept.
# relevant_pages = find_relevant_pages(
#     query_text="system design concepts",
#     book_title="Designing Data Intensive Applications",
# )
# relevant_pages

# Run the full RAG loop and inspect citations.
# result = answer_question(
#     query_text="What pages in this book discuss system design?",
#     book_title="Designing Data Intensive Applications",
# )
# print(result["answer"])
# pd.DataFrame(result["citations"])

In [17]:
# Example usage: run these lines step by step after you place PDFs under PROJECT_ROOT / 'books'.

client = connect_weaviate_local()

# Preview parsed chunk records before indexing.
sample_records = collect_chunk_records(PDF_ROOT)
# tmp add
sample_records = random.sample(sample_records,100)
# tmp add

# Build or rebuild the vector index.
index_preview = index_books(client = client, records = sample_records, pdf_root=PDF_ROOT, reset_collection=False)

Parsing PDFs with LangChain:   0%|          | 0/13 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Indexing chunks:   0%|          | 0/100 [00:00<?, ?it/s]

Indexed 100 chunks into collection 'BookChunks'.


In [18]:
# Ask which books discuss a topic across the whole library.
relevant_books = find_relevant_books(query_text = "Which books talk about China?")

# Ask which pages in one book mention a concept.
relevant_pages = find_relevant_pages(
    query_text="system design concepts",
    book_title="Machine Learning An Algorithmic Perspective",
)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
relevant_books

,book_title,book_category,match_count,best_distance
0,Thomas Calculus,Mathematics,3,0.524011
1,The Man Who Solved The Market,non-fiction,2,0.521394
2,Adaptive Markets Financial Evolution at the Sp...,non-fiction,2,0.556340
3,The Signal and the Noise Why So Many Predictio...,non-fiction,2,0.563601
4,Machine Learning An Algorithmic Perspective,Machine-Learning,1,0.557055
5,Fooled by Randomness The Hidden Role of Chan...,non-fiction,1,0.571099
6,The book of why the new science of cause and e...,non-fiction,1,0.576517


In [20]:
relevant_pages

,book_title,page_number,match_count,best_distance,sample_text
0,Machine Learning An Algorithmic Perspective,452,1,0.410295,Chapman & Hall/CRC Machine Learning & Pattern ...
1,Machine Learning An Algorithmic Perspective,315,1,0.476093,294 ■Machine Learning: An Algorithmic Perspect...
2,Machine Learning An Algorithmic Perspective,324,1,0.479371,. Modify it to use the SOM instead and compare...
3,Machine Learning An Algorithmic Perspective,133,1,0.508645,112 ■Machine Learning: An Algorithmic Perspect...
4,Machine Learning An Algorithmic Perspective,86,1,0.509182,. Computing the smallest value of this means d...
5,Machine Learning An Algorithmic Perspective,186,1,0.523294,Probabilistic Learning ■165 FIGURE 7.8 Two tes...


In [22]:
# print(len(sample_records))
# index = 77
# print('book_category: ', sample_records[index].book_category)
# print('book_id: ',sample_records[index].book_id)
# print('book_title: ',sample_records[index].book_title)
# print('source_path: ',sample_records[index].source_path)
# print('text: ',sample_records[index].text)
# print('chunk_index: ',sample_records[index].chunk_index)
# print('page_number: ',sample_records[index].page_number)

In [23]:
### CHECKING HOW MANY COLLECTIONS AND VECTORS IN WEAVIATE
client = weaviate.connect_to_local(host='127.0.0.1', port=8080)
pprint.pprint(client.collections.list_all())
bookchunk_collection = client.collections.get(COLLECTION_NAME)
vector_count = bookchunk_collection.aggregate.over_all(total_count=True)
print(f"Total vectors in collection '{COLLECTION_NAME}': {vector_count}")
client.close()

{'BookChunks': _CollectionConfigSimple(name='BookChunks',
                                       description=None,
                                       generative_config=None,
                                       properties=[_Property(name='book_id',
                                                             description=None,
                                                             data_type=<DataType.TEXT: 'text'>,
                                                             index_filterable=True,
                                                             index_range_filters=False,
                                                             index_searchable=True,
                                                             nested_properties=None,
                                                             tokenization=<Tokenization.WORD: 'word'>,
                                                             vectorizer_config=None,
                                         

In [12]:
# The code to delete the collection is provided here for convenience, but be cautious when running it as it will remove all indexed data.
# delete_collection(client, collection_name = COLLECTION_NAME)

Collection 'BookChunks' deleted.
